In [1]:
## 2026 AUSTRIAN GRAND PRIX — PODIUM PREDICTION
### Rolling data: R1 AUS + R2 CHN + R3 JPN + R4 MIA + R5 CAN + R6 MON + R7 BCN
### Austrian GP qualifying grid: June 27 2026 (Red Bull Ring, Spielberg)
### Weather: EXTREME HEAT ~35°C race day, dry — high tyre & brake degradation
###
### AUSTRIAN GP — CIRCUIT-SPECIFIC FACTORS (vs Barcelona model):
###   1. FP2 Long-Run Race Pace  (medium tyre stints from Friday)
###   2. Tyre degradation index  (rear-limited high-speed corners T3, T7, T9)
###   3. Brake degradation index (heavy braking into T3 & T4 — unique to RBR)
###   4. Power unit deployment   (Active aero / biturbo hybrid; high-altitude stress)
###   5. Overtaking index        (RBR avg 90 overtakes last 3 seasons — very high)
###   6. Pit stop score          (undercut opportunity strong at this circuit)
###   7. Grid weight = 0.15      (lowest of season — overtaking very common here)
###   8. VER crash risk flag     (crashed Q3 — car damage / confidence risk)
###   9. RUS yellow flag note    (pole under investigation — cleared by stewards)

import sys
!{sys.executable} -m pip install fastf1 xgboost --quiet

import fastf1
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

# ── Austrian GP Qualifying Grid (June 27 2026) ────────────────────────────────
# Source: formula1.com, total-motorsport.com, the-race.com, gpfans.com
# RUS pole: 1:06.113 | LEC +0.236 | HAM +0.295 | ANT +0.301 | VER +0.362
# NOR +0.389 | PIA +0.398 | HAD +0.519 | LAW +0.842 | LIN +0.894
# Q2 elim (P11-P16): GAS, BOR, BEA, HUL, OCO, COL
# Q1 elim (P17-P22): SAI, ALB, PER, BOT, ALO, STR
#
# NOTES:
#   RUS: pole cleared by stewards (yellow flag infringement — no further action)
#   VER: crashed at Turn 9 on final Q3 lap — car damage risk for race start
#   ANT: backed off on final Q3 lap due to yellow flags → starts P4, underperformed
#   LEC: Ferrari strong in Q3 — P2 is best quali result in 2026 so far
#   COL: P16 — alpine team-mate Gasly P11, ran wide at Turn 1 on last lap

austrian_quali_data = [
    {"Driver": "RUS", "BestLap_s": 66.113, "AustrianGrid": 1},
    {"Driver": "LEC", "BestLap_s": 66.349, "AustrianGrid": 2},
    {"Driver": "HAM", "BestLap_s": 66.408, "AustrianGrid": 3},
    {"Driver": "ANT", "BestLap_s": 66.414, "AustrianGrid": 4},
    {"Driver": "VER", "BestLap_s": 66.475, "AustrianGrid": 5},   # crashed, earlier lap used
    {"Driver": "NOR", "BestLap_s": 66.502, "AustrianGrid": 6},
    {"Driver": "PIA", "BestLap_s": 66.511, "AustrianGrid": 7},
    {"Driver": "HAD", "BestLap_s": 66.632, "AustrianGrid": 8},
    {"Driver": "LAW", "BestLap_s": 66.955, "AustrianGrid": 9},
    {"Driver": "LIN", "BestLap_s": 67.007, "AustrianGrid": 10},
    {"Driver": "GAS", "BestLap_s": 67.200, "AustrianGrid": 11},
    {"Driver": "BOR", "BestLap_s": 67.350, "AustrianGrid": 12},
    {"Driver": "BEA", "BestLap_s": 67.500, "AustrianGrid": 13},
    {"Driver": "HUL", "BestLap_s": 67.600, "AustrianGrid": 14},
    {"Driver": "OCO", "BestLap_s": 67.700, "AustrianGrid": 15},
    {"Driver": "COL", "BestLap_s": 67.800, "AustrianGrid": 16},
    {"Driver": "SAI", "BestLap_s": 68.100, "AustrianGrid": 17},
    {"Driver": "ALB", "BestLap_s": 68.150, "AustrianGrid": 18},
    {"Driver": "PER", "BestLap_s": 68.400, "AustrianGrid": 19},
    {"Driver": "BOT", "BestLap_s": 68.600, "AustrianGrid": 20},
    {"Driver": "ALO", "BestLap_s": 68.800, "AustrianGrid": 21},
    {"Driver": "STR", "BestLap_s": 69.000, "AustrianGrid": 22},
]

quali_df = pd.DataFrame(austrian_quali_data)
pole_time = quali_df["BestLap_s"].min()
quali_df["AustrianGapFromPole_s"]   = (quali_df["BestLap_s"] - pole_time).round(3)
quali_df["AustrianGapFromPolePct"] = (quali_df["AustrianGapFromPole_s"] / pole_time * 100).round(4)

# Race-day weather: June 28 2026, Spielberg, Austria
# Source: formula1.com, total-motorsport.com — European heatwave, ~35°C, ~52°C tarmac
rain_probability = 0.03
temperature      = 35.0   # EXTREME — heatwave across Europe
quali_df["RainProbability"] = rain_probability
quali_df["Temperature"]     = temperature

print(quali_df[["Driver", "BestLap_s", "AustrianGrid",
                "AustrianGapFromPole_s", "AustrianGapFromPolePct"]])
print(f"\nConditions: {temperature}°C | Rain: {rain_probability:.0%} | EUROPEAN HEATWAVE")


# ── Driver Championship Standings after Barcelona GP (R7) ─────────────────────
# Source: total-motorsport.com, si.com, racingnews365.com (after Colapinto penalty)
# ANT: 156 | HAM: 115 | RUS: 106 | LEC: 75 | NOR: 73 | PIA: 63
# VER: 55 | GAS: 47 | HAD: 38 | LAW: 33 | LIN: 20 | BEA: 18
# COL: 16 | SAI: 6 | ALB: 5 | OCO: 3 | BOR: 2 | ALO: 1
# HUL: 0 | BOT: 0 | PER: 0 | STR: 0
driver_points = {
    "ANT": 156, "HAM": 115, "RUS": 106, "LEC": 75,
    "NOR": 73,  "PIA": 63,  "VER": 55,  "GAS": 47,
    "HAD": 38,  "LAW": 33,  "LIN": 20,  "BEA": 18,
    "COL": 16,  "SAI": 6,   "ALB": 5,   "OCO": 3,
    "BOR": 2,   "ALO": 1,   "HUL": 0,   "BOT": 0,
    "PER": 0,   "STR": 0,
}

# Constructor standings after Barcelona GP (R7)
# MER: 262 | FER: 190 | MCL: 136 | RBR: 93 | ALP: 63
# RBU: 53 | HAA: 21 | WIL: 11 | AUD: 2 | AST: 1 | CAD: 0
team_points = {
    "Mercedes":      262,
    "Ferrari":       190,
    "McLaren":       136,
    "Red Bull":       93,
    "Alpine":         63,
    "Racing Bulls":   53,
    "Haas":           21,
    "Williams":       11,
    "Audi":            2,
    "Aston Martin":    1,
    "Cadillac":        0,
}

driver_to_team = {
    "RUS": "Mercedes",     "ANT": "Mercedes",
    "HAM": "Ferrari",      "LEC": "Ferrari",
    "NOR": "McLaren",      "PIA": "McLaren",
    "VER": "Red Bull",     "HAD": "Red Bull",
    "BEA": "Haas",         "OCO": "Haas",
    "LAW": "Racing Bulls", "LIN": "Racing Bulls",
    "HUL": "Audi",         "BOR": "Audi",
    "GAS": "Alpine",       "COL": "Alpine",
    "SAI": "Williams",     "ALB": "Williams",
    "PER": "Cadillac",     "BOT": "Cadillac",
    "ALO": "Aston Martin", "STR": "Aston Martin",
}

max_driver_pts = max(v for v in driver_points.values() if v > 0)
quali_df["DriverPoints"]     = quali_df["Driver"].map(driver_points).fillna(0)
quali_df["DriverPointsNorm"] = (quali_df["DriverPoints"] / max_driver_pts).round(4)


# ── Rolling Race Results R1-R7 ────────────────────────────────────────────────
aus_finish = {
    "RUS": 1,  "ANT": 2,  "LEC": 3,  "HAM": 4,
    "NOR": 5,  "VER": 6,  "BEA": 7,  "LIN": 8,
    "BOR": 9,  "GAS": 10, "OCO": 11, "ALB": 12,
    "LAW": 13, "COL": 14, "SAI": 15, "PER": 16,
    "STR": 17, "ALO": 18, "BOT": 19, "HAD": 20,
    "PIA": 21, "HUL": 22,
}

china_finish = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,
    "BEA": 5,  "GAS": 6,  "LAW": 7,  "HAD": 8,
    "SAI": 9,  "COL": 10, "HUL": 11, "LIN": 12,
    "BOT": 13, "OCO": 14, "PER": 15, "VER": 16,
    "ALO": 17, "STR": 18, "NOR": 19, "BOR": 20,
    "ALB": 21, "PIA": 22,
}

japan_finish = {
    "ANT": 1,  "PIA": 2,  "LEC": 3,  "RUS": 4,
    "NOR": 5,  "HAM": 6,  "GAS": 7,  "VER": 8,
    "LAW": 9,  "OCO": 10, "HUL": 11, "HAD": 12,
    "BOR": 13, "LIN": 14, "SAI": 15, "COL": 16,
    "PER": 17, "ALO": 18, "BOT": 19, "ALB": 20,
    "STR": 21, "BEA": 22,
}

miami_finish = {
    "ANT": 1,  "NOR": 2,  "PIA": 3,  "RUS": 4,
    "VER": 5,  "HAM": 6,  "COL": 7,  "LEC": 8,
    "SAI": 9,  "ALB": 10, "BEA": 11, "BOR": 12,
    "OCO": 13, "LIN": 14, "ALO": 15, "PER": 16,
    "STR": 17, "BOT": 18, "HUL": 19, "LAW": 20,
    "GAS": 21, "HAD": 22,
}

canada_finish = {
    "ANT": 1,  "HAM": 2,  "VER": 3,  "LEC": 4,
    "HAD": 5,  "COL": 6,  "LAW": 7,  "GAS": 8,
    "SAI": 9,  "BEA": 10, "PIA": 11, "HUL": 12,
    "BOR": 13, "OCO": 14, "STR": 15, "BOT": 16,
    "ALB": 17, "ALO": 18, "LIN": 19, "NOR": 20,
    "RUS": 21, "PER": 22,
}

monaco_finish = {
    "ANT": 1,  "HAM": 2,  "GAS": 3,  "HAD": 4,
    "PIA": 5,  "LAW": 6,  "LIN": 7,  "ALB": 8,
    "OCO": 9,  "ALO": 10, "BOR": 11, "RUS": 12,
    "HUL": 13, "COL": 14, "PER": 15,
    "VER": 17, "LEC": 18, "NOR": 19, "BOT": 20,
    "BEA": 21, "SAI": 22, "STR": 22,
}

# Barcelona GP final classification (June 14 2026)
# Source: formula1.com, the-race.com, total-motorsport.com, silverstone.co.uk
# HAM P1, RUS P2, NOR P3, VER P4, PIA P5, HAD P6, GAS P7, LAW P8, LIN P9
# COL P10 (after Colapinto penalty dropped to P10 from P9)
# DNFs: LEC (power steering), ANT (electrical/engine), BEA, ALB, ALO, HUL, BOT, STR
barcelona_finish = {
    "HAM": 1,  "RUS": 2,  "NOR": 3,  "VER": 4,
    "PIA": 5,  "HAD": 6,  "GAS": 7,  "LAW": 8,
    "LIN": 9,  "COL": 10, "BOR": 11, "SAI": 12,
    "OCO": 13, "PER": 14,
    "LEC": 16, "ANT": 17, "BEA": 18, "ALB": 19,
    "ALO": 20, "HUL": 21, "BOT": 22, "STR": 22,
}

# Qualifying grids for all 7 previous rounds
aus_grid = {
    "RUS": 1,  "ANT": 2,  "HAD": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "HAM": 7,  "LAW": 8,  "LIN": 9,  "BOR": 10,
    "OCO": 11, "HUL": 12, "ALB": 13, "GAS": 14, "COL": 15,
    "BEA": 16, "ALO": 17, "PER": 18, "BOT": 19, "VER": 20,
    "SAI": 21, "STR": 22,
}

china_grid = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "GAS": 7,  "VER": 8,  "HAD": 9,  "BEA": 10,
    "HUL": 11, "COL": 12, "OCO": 13, "LAW": 14, "LIN": 15,
    "BOR": 16, "SAI": 17, "ALB": 18, "ALO": 19, "BOT": 20,
    "STR": 21, "PER": 22,
}

japan_grid = {
    "ANT": 1,  "RUS": 2,  "PIA": 3,  "LEC": 4,  "NOR": 5,
    "HAM": 6,  "GAS": 7,  "HAD": 8,  "BOR": 9,  "LIN": 10,
    "VER": 11, "OCO": 12, "HUL": 13, "LAW": 14, "COL": 15,
    "SAI": 16, "ALB": 17, "BEA": 18, "PER": 19, "BOT": 20,
    "ALO": 21, "STR": 22,
}

miami_grid = {
    "ANT": 1,  "NOR": 2,  "VER": 3,  "LEC": 4,  "PIA": 5,
    "RUS": 6,  "GAS": 7,  "BEA": 8,  "SAI": 9,  "OCO": 10,
    "HUL": 11, "LAW": 12, "LIN": 13, "ALB": 14, "BOR": 15,
    "COL": 16, "ALO": 17, "STR": 18, "BOT": 19, "HAM": 20,
    "HAD": 21, "PER": 22,
}

canada_grid = {
    "RUS": 1,  "ANT": 2,  "NOR": 3,  "PIA": 4,  "HAM": 5,
    "VER": 6,  "HAD": 7,  "LEC": 8,  "LIN": 9,  "COL": 10,
    "HUL": 11, "LAW": 12, "BOR": 13, "GAS": 14, "SAI": 15,
    "BEA": 16, "OCO": 17, "ALB": 18, "ALO": 19, "PER": 20,
    "BOT": 21, "STR": 22,
}

monaco_grid = {
    "ANT": 1,  "VER": 2,  "HAM": 3,  "LEC": 4,  "HAD": 5,
    "RUS": 6,  "PIA": 7,  "NOR": 8,  "GAS": 9,  "LAW": 10,
    "ALB": 11, "SAI": 12, "HUL": 13, "COL": 14, "LIN": 15,
    "BOR": 16, "OCO": 17, "PER": 18, "BEA": 19, "BOT": 20,
    "ALO": 21, "STR": 22,
}

barcelona_grid = {
    "RUS": 1,  "HAM": 2,  "ANT": 3,  "NOR": 4,  "VER": 5,
    "HAD": 6,  "PIA": 7,  "LAW": 8,  "HUL": 9,  "LEC": 10,
    "LIN": 11, "BOR": 12, "COL": 13, "GAS": 14, "BEA": 15,
    "SAI": 16, "OCO": 17, "ALB": 18, "PER": 19, "BOT": 20,
    "STR": 21, "ALO": 22,
}

austrian_grid = {
    "RUS": 1,  "LEC": 2,  "HAM": 3,  "ANT": 4,  "VER": 5,
    "NOR": 6,  "PIA": 7,  "HAD": 8,  "LAW": 9,  "LIN": 10,
    "GAS": 11, "BOR": 12, "BEA": 13, "HUL": 14, "OCO": 15,
    "COL": 16, "SAI": 17, "ALB": 18, "PER": 19, "BOT": 20,
    "ALO": 21, "STR": 22,
}

# ── Build 7-race rolling features ─────────────────────────────────────────────
drivers = list(driver_to_team.keys())
N = len(drivers)

records = []
for drv in drivers:
    fp = [
        aus_finish.get(drv, 20),
        china_finish.get(drv, 20),
        japan_finish.get(drv, 20),
        miami_finish.get(drv, 20),
        canada_finish.get(drv, 20),
        monaco_finish.get(drv, 20),
        barcelona_finish.get(drv, 20),
    ]
    gp = [
        aus_grid.get(drv, 20),
        china_grid.get(drv, 20),
        japan_grid.get(drv, 20),
        miami_grid.get(drv, 20),
        canada_grid.get(drv, 20),
        monaco_grid.get(drv, 20),
        barcelona_grid.get(drv, 20),
    ]
    aut_gp = austrian_grid.get(drv, 20)

    avg_finish_norm = round(sum((f-1)/(N-1) for f in fp) / 7, 4)
    avg_grid_norm   = round(sum((g-1)/(N-1) for g in gp) / 7, 4)

    # Exponential decay: AUT quali 30%, BCN race 22%, MON 15%, CAN 12%, MIA 8%, JPN 6%, CHN 4%, AUS 3%
    recent_finish_norm = round(
        0.30*(aut_gp-1)/(N-1) +       # Austrian qualifying as most recent signal
        0.22*(fp[6]-1)/(N-1) +         # Barcelona race
        0.15*(fp[5]-1)/(N-1) +         # Monaco
        0.12*(fp[4]-1)/(N-1) +         # Canada
        0.08*(fp[3]-1)/(N-1) +         # Miami
        0.06*(fp[2]-1)/(N-1) +         # Japan
        0.04*(fp[1]-1)/(N-1) +         # China
        0.03*(fp[0]-1)/(N-1), 4        # Australia
    )

    # DNF rate (7 races — includes BCN high DNF rate)
    dnf_thresholds = [18, 16, 21, 19, 20, 16, 15]
    dnf_flags = [1 if fp[i] >= dnf_thresholds[i] else 0 for i in range(7)]
    dnf_rate = round(sum(dnf_flags) / 7, 3)

    records.append({
        "Driver":           drv,
        "Team":             driver_to_team[drv],
        "AusFinish":        fp[0], "ChinaFinish":     fp[1],
        "JapanFinish":      fp[2], "MiamiFinish":     fp[3],
        "CanadaFinish":     fp[4], "MonacoFinish":    fp[5],
        "BarcelonaFinish":  fp[6],
        "AvgFinishNorm":    avg_finish_norm,
        "RecentFinishNorm": recent_finish_norm,
        "AvgGridNorm":      avg_grid_norm,
        "AustrianGrid":     aut_gp,
        "DNF_rate":         dnf_rate,
    })

rolling_df = pd.DataFrame(records)
max_team_pts   = max(team_points.values())

rolling_df["TeamScore"] = rolling_df["Team"].map(
    {t: max(p, 0.5) / max_team_pts for t, p in team_points.items()}
).round(4)
rolling_df["DriverPointsNorm"] = rolling_df["Driver"].map(
    {d: p / max_driver_pts for d, p in driver_points.items()}
).round(4)

print("\n" + "=" * 120)
print("  ROLLING FEATURES — AUSTRIAN GP (R1-R7 inclusive)")
print("=" * 120)


# ── AUSTRIAN GP FEATURE 1: FP2 Long-Run Race Pace ────────────────────────────
# Source: the-race.com, autosport.com, total-motorsport.com, thejudge13.com
# ALL ON MEDIUMS (C4):
# 1. RUS   1:11.220 (4 laps)
# 2. ANT   1:11.265 (6 laps)
# 3. NOR   1:11.466 (8 laps)
# 4. VER   1:11.704 (9 laps)
# 5. HAM   1:11.773 (6 laps)
# 6. PIA   1:11.777 (8 laps)
# 7. HAD   1:11.910 (5 laps)
# 8. LEC   1:12.116 (6 laps)
# HUL (Audi): 1:12.134 on HARDS (9 laps) — equivalent ~1:12.0 on mediums
# GAS/COL (Alpine): comparable to Haas/RBu bracket
# Cadillac: no representative data (PER/BOT mechanical issues in FP2)

fp2_long_run_data = [
    # Gap to Russell benchmark (0 = fastest)
    {"Driver": "RUS", "FP2_LongRun_GapToFastest_s": 0.000},   # 1:11.220 — 4 laps
    {"Driver": "ANT", "FP2_LongRun_GapToFastest_s": 0.045},   # 1:11.265 — 6 laps
    {"Driver": "NOR", "FP2_LongRun_GapToFastest_s": 0.246},   # 1:11.466 — 8 laps (very stable)
    {"Driver": "VER", "FP2_LongRun_GapToFastest_s": 0.484},   # 1:11.704 — 9 laps (driveability issues T3)
    {"Driver": "HAM", "FP2_LongRun_GapToFastest_s": 0.553},   # 1:11.773 — 6 laps
    {"Driver": "PIA", "FP2_LongRun_GapToFastest_s": 0.557},   # 1:11.777 — 8 laps (variable)
    {"Driver": "HAD", "FP2_LongRun_GapToFastest_s": 0.690},   # 1:11.910 — 5 laps
    {"Driver": "LEC", "FP2_LongRun_GapToFastest_s": 0.896},   # 1:12.116 — 6 laps (detuned PU test)
    {"Driver": "HUL", "FP2_LongRun_GapToFastest_s": 0.914},   # 1:12.134 on hards → adj ~1:12.03 mediums
    {"Driver": "BOR", "FP2_LongRun_GapToFastest_s": 1.050},   # Audi pair estimate
    {"Driver": "LAW", "FP2_LongRun_GapToFastest_s": 1.100},   # Racing Bulls
    {"Driver": "LIN", "FP2_LongRun_GapToFastest_s": 1.150},
    {"Driver": "GAS", "FP2_LongRun_GapToFastest_s": 1.200},   # Alpine
    {"Driver": "COL", "FP2_LongRun_GapToFastest_s": 1.350},
    {"Driver": "BEA", "FP2_LongRun_GapToFastest_s": 1.300},   # Haas
    {"Driver": "OCO", "FP2_LongRun_GapToFastest_s": 1.400},
    {"Driver": "SAI", "FP2_LongRun_GapToFastest_s": 1.800},   # Williams
    {"Driver": "ALB", "FP2_LongRun_GapToFastest_s": 1.850},
    {"Driver": "ALO", "FP2_LongRun_GapToFastest_s": 2.200},   # Aston Martin
    {"Driver": "STR", "FP2_LongRun_GapToFastest_s": 2.350},
    {"Driver": "PER", "FP2_LongRun_GapToFastest_s": 2.500},   # Cadillac — no FP2 data; estimated
    {"Driver": "BOT", "FP2_LongRun_GapToFastest_s": 2.600},   # Cadillac — no FP2 data; estimated
]

fp2_df = pd.DataFrame(fp2_long_run_data)
print("\nFP2 Long-Run Pace (medium-equivalent gap to Russell — 1:11.220):")
print(fp2_df.sort_values("FP2_LongRun_GapToFastest_s").to_string(index=False))


# ── AUSTRIAN GP FEATURE 2: Tyre Degradation Index ────────────────────────────
# Red Bull Ring-specific: rear-limited T3 (Turn 3 long RH corner), T7 (fast left),
# T9 (where VER crashed in quali). High-speed sustained loading on rear tyres.
# Different profile to Barcelona (front-left biased).
# 35°C air + 52°C tarmac = EXTREME degradation expected (tyre choice: M+H focus).
# McLaren traditionally GOOD in hot conditions on this circuit.
# Source: autosport.com, the-race.com (FP2 analysis), historical RBR deg data

tyre_deg_austria = {
    "Mercedes":      0.18,   # excellent thermal management; W16 aero efficient
    "McLaren":       0.20,   # very good in HOT conditions historically — RBR suits MCL
    "Red Bull":      0.28,   # RBR upgrade: T3 exit driveability issues = tyre stress
    "Racing Bulls":  0.32,
    "Ferrari":       0.35,   # new PU upgrade, cooling stress at altitude; deg concern
    "Alpine":        0.42,
    "Haas":          0.44,
    "Audi":          0.40,   # Hulkenberg strong on hards in FP2
    "Williams":      0.50,
    "Aston Martin":  0.58,
    "Cadillac":      0.55,
}

rolling_df["TyreDegIndex"] = rolling_df["Team"].map(tyre_deg_austria).round(3)


# ── AUSTRIAN GP FEATURE 3: Brake Degradation Index (UNIQUE to RBR) ───────────
# Red Bull Ring has 3 heavy braking events (T3, T4, T10) on a SHORT 4.318km lap.
# 71 laps = VERY high total braking load. High altitude + 35°C = brake cooling stress.
# Leclerc noted brake sensitivity; Piastri complained "like a lottery" in FP1.
# Brake reliability is a serious risk factor for the full 71-lap race.
# Scale: 0 = most reliable brakes | 1 = highest brake deg risk

brake_deg_austria = {
    "Mercedes":      0.20,   # strong brake management historically
    "Ferrari":       0.38,   # both drivers noted brake issues in practice
    "McLaren":       0.32,   # Piastri reported brake issues in FP1
    "Red Bull":      0.35,
    "Racing Bulls":  0.40,
    "Alpine":        0.45,
    "Haas":          0.48,
    "Audi":          0.42,
    "Williams":      0.52,
    "Cadillac":      0.58,   # reliability nightmare in FP1+FP2
    "Aston Martin":  0.55,
}

rolling_df["BrakeDegIndex"] = rolling_df["Team"].map(brake_deg_austria).round(3)


# ── AUSTRIAN GP FEATURE 4: Power Unit Deployment Score ───────────────────────
# RBR is high-altitude (660m) + active aero / biturbo hybrid.
# Battery thermal management critical — hot ambient temps reduce cooling margins.
# Ferrari's new ADUO-assisted PU upgrade (first race with full spec).
# Mercedes PU benchmark (battery upgrade for Austria per The Race report).
# Scale: 1 = best PU/energy deployment | 0 = weakest

pu_deployment_score = {
    "Mercedes":      0.95,   # battery upgrade for Austria; benchmark PU
    "Ferrari":       0.82,   # ADUO upgrade, new cylinder block — cleaner burn at heat
    "McLaren":       0.88,   # strong energy management, Mercedes PU customer
    "Red Bull":      0.75,   # T3 driveability issues with deployment (Mekies confirmed)
    "Racing Bulls":  0.72,   # Honda PU (Red Bull spec)
    "Alpine":        0.70,   # Renault PU
    "Haas":          0.68,   # Ferrari PU customer
    "Audi":          0.65,
    "Williams":      0.70,   # Mercedes PU customer
    "Cadillac":      0.55,   # ECU issues in both FP1/FP2; reliability risk
    "Aston Martin":  0.60,   # Mercedes PU but slow chassis
}

rolling_df["PUDeploymentScore"] = rolling_df["Team"].map(pu_deployment_score).round(3)


# ── AUSTRIAN GP FEATURE 5: Overtaking Potential Index ────────────────────────
# Red Bull Ring averaged 90 overtakes over last 3 seasons — highest on calendar.
# DRS zones on main straight + braking into T3/T4.
# Starting position much less decisive here than Monaco or even Barcelona.
# Recovery from poor grid positions very achievable.

overtake_index = {
    "HAM": 0.93,   # best overtaker on grid; from P3, strong race craft
    "ANT": 0.89,   # from P4 — likely to move forward; dominant season pace
    "VER": 0.85,   # from P5 — always dangerous in race; Q3 crash risk concern
    "NOR": 0.82,   # strong overtaker; McLaren good in hot races
    "PIA": 0.79,
    "LEC": 0.78,   # from P2 — may fade on brakes but solid racer
    "RUS": 0.76,
    "HAD": 0.73,
    "LAW": 0.70,
    "GAS": 0.68,
    "COL": 0.64,
    "LIN": 0.66,
    "HUL": 0.63,
    "BEA": 0.58,
    "SAI": 0.72,   # experienced — good at making passes
    "ALB": 0.62,
    "BOR": 0.57,
    "OCO": 0.61,
    "PER": 0.55,
    "BOT": 0.50,
    "ALO": 0.74,   # experience / racecraft at RBR
    "STR": 0.44,
}

rolling_df["OvertakeIndex"] = rolling_df["Driver"].map(overtake_index).fillna(0.55).round(3)


# ── AUSTRIAN GP FEATURE 6: Pit Stop Score ────────────────────────────────────
# Undercut is powerful at RBR — 4.3km lap means small time gaps amplified.
# A 2-3 second stop time difference = significant strategic advantage.
# Cadillac have had reliability/pit issues. Red Bull home race = maximum motivation.

pitstop_score = {
    "Mercedes":      2.30,
    "Ferrari":       2.42,
    "McLaren":       2.38,
    "Red Bull":      2.45,   # home race — motivated but upgrade complexity
    "Racing Bulls":  2.55,
    "Alpine":        2.62,
    "Haas":          2.68,
    "Audi":          2.70,
    "Williams":      2.78,
    "Cadillac":      3.10,   # nightmare reliability in practice; caution
    "Aston Martin":  2.90,
}

rolling_df["PitStopScore"] = rolling_df["Team"].map(pitstop_score).round(3)


# ── AUSTRIAN GP FEATURE 7: VER Crash Risk Flag ────────────────────────────────
# Verstappen crashed at Turn 9 in Q3. Team confirmed: "lost aero performance on rear"
# (sidepod/floor damage from upgrade). Risk: car may not be fully repaired / balanced.
# This is a SPECIFIC one-race risk factor for VER only.
# Source: the-race.com (Mekies: "we lost aero performance on the rear of the car")

rolling_df["QualCrashRiskFlag"] = rolling_df["Driver"].map(
    {d: 1 if d == "VER" else 0 for d in drivers}
).fillna(0)


# ── Merge all data ────────────────────────────────────────────────────────────
df = quali_df.merge(
    rolling_df[[
        "Driver", "Team", "AvgFinishNorm", "RecentFinishNorm", "AvgGridNorm",
        "DNF_rate", "TeamScore", "TyreDegIndex", "BrakeDegIndex",
        "PUDeploymentScore", "OvertakeIndex", "PitStopScore",
        "QualCrashRiskFlag"
    ]],
    on="Driver", how="left"
)

df_final = df.merge(
    fp2_df[["Driver", "FP2_LongRun_GapToFastest_s"]],
    on="Driver", how="left"
)


# ── Feature Engineering ───────────────────────────────────────────────────────

# Tyre + Brake composite (lower = better Austria race performance)
df_final["TyreBrakeComposite"] = (
    df_final["TyreDegIndex"] * 0.45 +
    df_final["BrakeDegIndex"] * 0.35 +
    (df_final["PitStopScore"] / df_final["PitStopScore"].max()) * 0.20
).round(4)

# Race pace composite (FP2 long run + tyre/brake deg + PU deployment)
max_fp2 = df_final["FP2_LongRun_GapToFastest_s"].max()
df_final["RacePaceComposite"] = (
    (df_final["FP2_LongRun_GapToFastest_s"] / max_fp2) * 0.55 +
    df_final["TyreDegIndex"] * 0.25 +
    (1 - df_final["PUDeploymentScore"]) * 0.20
).round(4)

# Quali advantage adjusted for overtaking potential
# (grid position matters LESS at RBR — weight reduced)
df_final["QualiRaceAdjusted"] = (
    (df_final["AustrianGapFromPolePct"] / df_final["AustrianGapFromPolePct"].max()) * 0.60 +
    (1 - df_final["OvertakeIndex"]) * 0.25 +
    df_final["QualCrashRiskFlag"] * 0.15   # VER crash damage risk penalty
).round(4)

feature_cols = [
    # Qualifying
    "AustrianGapFromPole_s",     # Gap to pole
    "AustrianGrid",              # Grid position (LOW weight — 90 overtakes at RBR)
    "QualiRaceAdjusted",         # Quali gap adjusted for overtaking + crash risk

    # FP2 Race pace (critical — confirmed by teams post-FP2)
    "FP2_LongRun_GapToFastest_s",
    "RacePaceComposite",         # Race pace + deg + PU combined

    # Austria-specific physical factors
    "TyreDegIndex",              # Rear-limited: T3, T7, T9
    "BrakeDegIndex",             # RBR unique: heavy braking T3/T4/T10, 71 laps
    "TyreBrakeComposite",        # Combined physical stress
    "PUDeploymentScore",         # Energy/hybrid deployment at altitude + heat
    "OvertakeIndex",             # Very high — RBR avg 90 overtakes

    # Pit stop
    "PitStopScore",              # Undercut powerful at 4.3km circuit

    # Championship & constructor
    "TeamScore",
    "DriverPointsNorm",

    # Special flags
    "QualCrashRiskFlag",         # VER Q3 crash — potential race-start uncertainty

    # Rolling form
    "AvgFinishNorm",
    "RecentFinishNorm",
    "AvgGridNorm",
    "DNF_rate",

    # Weather
    "RainProbability",
    "Temperature",
]


# ── Target Variable ───────────────────────────────────────────────────────────
# Austrian GP: grid position LEAST important of all circuits visited so far.
# Race pace + physical management dominant. Overtaking very common.
# Grid weight: 0.15 (lowest of season)
df_final["RaceScore"] = (
    0.38 * df_final["AvgFinishNorm"] +
    0.30 * df_final["RecentFinishNorm"] +
    0.15 * df_final["AvgGridNorm"] +
    0.12 * df_final["TyreBrakeComposite"] +   # Austria-specific physical stress
    0.05 * (1 - df_final["PUDeploymentScore"])  # PU heat/altitude penalty
).round(4)

X = df_final[feature_cols].copy()
y = df_final["RaceScore"]

imputer   = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)


# ── Multi-model LOO-CV ────────────────────────────────────────────────────────
models = {
    "Ridge":         Ridge(alpha=1.0),
    "ElasticNet":    ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest":  RandomForestRegressor(
                         n_estimators=200, max_depth=4,
                         min_samples_leaf=3, random_state=42),
    "GradientBoost": GradientBoostingRegressor(
                         n_estimators=200, learning_rate=0.04,
                         max_depth=3, subsample=0.8, random_state=42),
    "XGBoost":       XGBRegressor(
                         n_estimators=200, learning_rate=0.04,
                         max_depth=3, subsample=0.8,
                         colsample_bytree=0.8, reg_lambda=2.5,
                         random_state=42),
}

loo     = LeaveOneOut()
results = {}

for name, mdl in models.items():
    pipe = (Pipeline([("scaler", StandardScaler()), ("model", mdl)])
            if name in ["Ridge", "ElasticNet"]
            else Pipeline([("model", mdl)]))
    errors = []
    for train_idx, test_idx in loo.split(X_imputed):
        pipe.fit(X_imputed[train_idx], y.iloc[train_idx])
        errors.append(abs(pipe.predict(X_imputed[test_idx])[0] - y.iloc[test_idx].iloc[0]))
    results[name] = {"MAE": round(np.mean(errors), 4), "StdDev": round(np.std(errors), 4)}

print(f"\n{'Model':<18} {'LOO MAE':>8} {'Std Dev':>8}  {'Verdict'}")
print("─" * 55)
for name, res in sorted(results.items(), key=lambda x: x[1]["MAE"]):
    best = " ← best" if res["MAE"] == min(r["MAE"] for r in results.values()) else ""
    print(f"{name:<18} {res['MAE']:>8.4f} {res['StdDev']:>8.4f}  {best}")


# ── Best model final prediction ───────────────────────────────────────────────
best_name  = min(results, key=lambda x: results[x]["MAE"])
best_model = models[best_name]
final_pipe = (Pipeline([("scaler", StandardScaler()), ("model", best_model)])
              if best_name in ["Ridge", "ElasticNet"]
              else Pipeline([("model", best_model)]))
final_pipe.fit(X_imputed, y)
df_final["PredictedScore"] = final_pipe.predict(X_imputed)

final = df_final.sort_values("PredictedScore").reset_index(drop=True)

print("\n" + "=" * 82)
print("   PREDICTED FINISHING ORDER — 2026 AUSTRIAN GRAND PRIX")
print("   7-race rolling model + Austrian-specific: brakes, PU altitude, tyre deg")
print("=" * 82)
print(f"  {'P':<4}{'DRV':<6}{'Team':<22}{'AUT Grid':<10}{'Score'}")
print("  " + "─" * 58)
for i, row in final.iterrows():
    crash_flag = " ⚠" if row["Driver"] == "VER" else ""
    print(f"  {i+1:<4}{row['Driver']:<6}{row['Team']:<22}"
          f"P{int(row['AustrianGrid']):<9}{row['PredictedScore']:.4f}{crash_flag}")

podium = final.head(3)
print(f"\n  {'='*62}")
print(f"  🥇 P1: {podium.iloc[0]['Driver']} ({podium.iloc[0]['Team']})")
print(f"  🥈 P2: {podium.iloc[1]['Driver']} ({podium.iloc[1]['Team']})")
print(f"  🥉 P3: {podium.iloc[2]['Driver']} ({podium.iloc[2]['Team']})")
print(f"\n  Best model:   {best_name}")
print(f"  LOO MAE:      {results[best_name]['MAE']:.4f}")
print(f"  Weather:      {df_final['Temperature'].iloc[0]:.0f}°C | "
      f"Rain: {df_final['RainProbability'].iloc[0]:.0%} | HEATWAVE")
print(f"  Grid weight:  0.15 (LOWEST of season — avg 90 overtakes at RBR)")
print(f"  Key factors:  FP2 long-run pace | Brake deg (T3/T4/T10, 71 laps)")
print(f"                PU altitude stress | Tyre rear-loading T3/T7/T9")
print(f"  VER ⚠ note:  Q3 crash — rear aero damage (team confirmed)")
print(f"                car repaired overnight but confidence/balance risk")
print(f"  ANT note:     Backed off on final Q3 lap (yellows) — true pace P3-P4")
print(f"                Championship leader; dominant race pace all season")
print(f"  LEC note:     P2 on grid — Ferrari's best quali in 2026; brake risk")
print(f"  MCL note:     Strong in HOT races — Norris sector 2 best in quali")
print(f"  {'='*62}")


# ── Feature importances ───────────────────────────────────────────────────────
if best_name in ["RandomForest", "GradientBoost", "XGBoost"]:
    importances = final_pipe.named_steps["model"].feature_importances_
    print("\nFeature importances:")
    for feat, imp in sorted(zip(feature_cols, importances), key=lambda x: -x[1]):
        bar = "█" * int(imp * 60)
        print(f"  {feat:<38} {imp:.4f}  {bar}")
elif best_name in ["Ridge", "ElasticNet"]:
    coefficients = final_pipe.named_steps["model"].coef_
    print("\nFeature coefficients:")
    for feat, coef in sorted(zip(feature_cols, coefficients), key=lambda x: -abs(x[1])):
        print(f"  {feat:<38} {coef:+.4f}")


# ── All-models side-by-side ───────────────────────────────────────────────────
all_predictions = {}
for name, mdl in models.items():
    pipe = (Pipeline([("scaler", StandardScaler()), ("model", mdl)])
            if name in ["Ridge", "ElasticNet"]
            else Pipeline([("model", mdl)]))
    pipe.fit(X_imputed, y)
    ranked = pd.Series(pipe.predict(X_imputed),
                       index=df_final["Driver"]).rank(method="min").astype(int)
    all_predictions[name] = ranked

pred_df = pd.DataFrame(all_predictions)
pred_df["BestModelPos"] = pd.Series(
    final_pipe.predict(X_imputed), index=df_final["Driver"]
).rank(method="min").astype(int)
pred_df = pred_df.sort_values("BestModelPos")

print("\n" + "=" * 112)
print("   PREDICTED FINISHING ORDER — 2026 AUSTRIAN GP  (all models)")
print("=" * 112)
col_w  = 14
header = f"  {'DRV':<6}{'Team':<22}"
for name in models: header += f"{name:>{col_w}}"
header += f"  {'★ Best':>{col_w}}"
print(header)
print("  " + "─" * 104)
for driver in pred_df.index:
    team = df_final.loc[df_final["Driver"] == driver, "Team"].values[0]
    row  = f"  {driver:<6}{team:<22}"
    for name in models:
        row += f"  P{pred_df.loc[driver, name]:<{col_w-2}}"
    row += f"  P{pred_df.loc[driver, 'BestModelPos']:<{col_w-2}}"
    if driver == "VER":
        row += " ⚠"
    print(row)

# ── Consensus podium ──────────────────────────────────────────────────────────
best_order = pred_df.sort_values("BestModelPos")
p1, p2, p3 = best_order.index[0], best_order.index[1], best_order.index[2]
t1 = df_final.loc[df_final["Driver"] == p1, "Team"].values[0]
t2 = df_final.loc[df_final["Driver"] == p2, "Team"].values[0]
t3 = df_final.loc[df_final["Driver"] == p3, "Team"].values[0]

print(f"\n  {'='*68}")
print(f"  Best model: {best_name} (LOO MAE: {results[best_name]['MAE']:.4f})")
print(f"  🥇 P1: {p1} ({t1})")
print(f"  🥈 P2: {p2} ({t2})")
print(f"  🥉 P3: {p3} ({t3})")
print(f"  Weather:    {df_final['Temperature'].iloc[0]:.0f}°C | "
      f"Rain: {df_final['RainProbability'].iloc[0]:.0%} | European heatwave")

print("\n  CONSENSUS PODIUM (across all models):")
print("  " + "─" * 48)
medals = ["🥇", "🥈", "🥉"]
for pos in [1, 2, 3]:
    pos_counts = {}
    for name in models:
        for d in pred_df[pred_df[name] == pos].index:
            pos_counts[d] = pos_counts.get(d, 0) + 1
    if pos_counts:
        top   = max(pos_counts, key=pos_counts.get)
        votes = pos_counts[top]
        print(f"  {medals[pos-1]} P{pos}: {top} — {votes}/{len(models)} models agree")
print(f"  {'='*68}")

print("""
  ┌──────────────────────────────────────────────────────────────────────┐
  │  AUSTRIAN GP CIRCUIT-SPECIFIC FACTORS SUMMARY                        │
  ├──────────────────────────────────────────────────────────────────────┤
  │  FP2 Long Run:   RUS 1:11.220 → ANT +0.045 → NOR +0.246 (mediums)  │
  │                  McLaren VERY stable over 8 laps — hot race suits    │
  │  Tyre Deg:       Rear-limited (T3, T7, T9). Mercedes best, MCL good │
  │  Brake Deg:      T3/T4/T10 heavy braking, 71 laps. Ferrari risk     │
  │                  (LEC noted brake issues). Cadillac major concern    │
  │  PU/Altitude:    660m altitude + 35°C. Mercedes battery upgrade ✓   │
  │                  Ferrari ADUO PU upgrade — first full race with spec │
  │                  Red Bull T3 deployment issues confirmed by Mekies   │
  │  Overtaking:     RBR avg 90 overtakes — grid pos LOW importance     │
  │                  Undercut strategy very powerful (short 4.3km lap)  │
  │  VER ⚠ :        Q3 crash at T9 — rear aero damage; car rebuilt      │
  │  ANT key fact:   True pace P3-4; expect to move forward from P4     │
  │  MCL hot race:   Norris historically thrives in heat — watch NOR    │
  └──────────────────────────────────────────────────────────────────────┘
""")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.6/427.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
   Driver  BestLap_s  AustrianGrid  AustrianGapFromPole_s  \
0     RUS     66.113             1                  0.000   
1     LEC     66.349             2                  0.236   
2     HAM     66.408             3                  0.295   
3     ANT     66.414             4                  0.301   
4     VER     66.475             5                  0.362   
5     NOR     66.502             6                  0.389   
6     PIA     66.511             7                  0.398   
7     HAD     66.632             8